# Solutions · 19 · Project: staffing a call centre from data

Worked solutions to the exercises of [notebook 19](../notebooks/19_project_staffing_a_call_centre.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

In [2]:
from engstoch import poisson as pp, queues as qu, des, montecarlo as mc, models, datasets
cc = pd.read_csv(datasets.path("call_centre.csv"), comment="#")
calls = cc[cc.handle_min > 0]
edges = np.arange(8.0, 20.01, 0.5); mid = edges[:-1] + 0.25
def slot_rates(df, exclude_outage=True):
    counts = []
    for d, g in df.groupby("day"):
        c = pp.counts_in_windows(g.arrival_h.to_numpy(), edges).astype(float)
        if exclude_outage and d == 8:
            c[(edges[:-1] >= 13.0) & (edges[:-1] < 14.5)] = np.nan
        counts.append(c)
    counts = np.array(counts)
    return np.nansum(counts, axis=0) / np.sum(~np.isnan(counts), axis=0) / 0.5
lam_week = slot_rates(calls[calls.weekday != "Mon"])
h = calls.handle_min.to_numpy(); ln_mu, ln_sd = np.log(h).mean(), np.log(h).std(ddof=1); mean_h = h.mean() / 60
target, answer = 0.8, 20 / 3600
def agents_fn(schedule):
    return lambda t: int(schedule[min(max(int((t - 8.0) / 0.5), 0), len(schedule) - 1)])
def evaluate(schedule, lam, reps=30, seed0=0, patience=lambda g: g.exponential(3.0 / 60)):
    rate_fn = lambda t: float(np.interp(t, mid, lam)) if 8.0 <= t < 20.0 else 0.0
    out = []
    for k in range(reps):
        r = des.call_centre(rate_fn, 1.05 * lam.max(), lambda g: float(np.exp(ln_mu + ln_sd * g.standard_normal())) / 60, agents_fn(schedule), 21.0, R(seed0 + k), patience_sampler=patience, answer_within=answer)
        slot = np.clip(((r["arrival"] - 8.0) / 0.5).astype(int), 0, len(mid) - 1)
        ok = (~r["abandoned"]) & (np.nan_to_num(r["wait"], nan=1.0) <= answer)
        out.append(([ok[slot == j].mean() for j in range(len(mid))], ok.mean(), r["abandoned"].mean()))
    sl = np.array([o[0] for o in out])
    return {"slot_sl": sl.mean(axis=0), "slot_se": sl.std(axis=0, ddof=1) / np.sqrt(reps), "day_sl": float(np.mean([o[1] for o in out])), "day_ab": float(np.mean([o[2] for o in out]))}
def lag_schedule(lam):
    from scipy import stats
    tgrid = np.linspace(8.0, 20.0, 2401); dt = tgrid[1] - tgrid[0]
    lam_t = np.interp(tgrid, mid, lam)
    surv = stats.lognorm.sf(np.arange(0, 2.0, dt) * 60, ln_sd, scale=np.exp(ln_mu))
    load = np.convolve(lam_t, surv)[: len(tgrid)] * dt
    return np.array([qu.staffing(load[(tgrid >= t0) & (tgrid < t0 + 0.5)].max() / mean_h, 1 / mean_h, target, answer)["c"] for t0 in edges[:-1]])

## Exercise 1

Build the Monday schedule from the Monday data alone (four days) and from the Tuesday-Friday profile scaled by the estimated Monday factor. Which is more reliable, and how do they compare in simulation?

In [3]:
lam_mon_direct = slot_rates(calls[calls.weekday == "Mon"])
week_days = calls[calls.weekday != "Mon"].groupby("day").size().drop(8)
factor = calls[calls.weekday == "Mon"].groupby("day").size().mean() / week_days.mean()
lam_mon_scaled = lam_week * factor
print(f"Monday factor {factor:.3f} (s.e. about {np.sqrt(1 / (4 * 1100) + 1 / (15 * 1080)):.3f})")
s_direct, s_scaled = lag_schedule(lam_mon_direct), lag_schedule(lam_mon_scaled)
print("direct  :", s_direct.tolist(), 0.5 * s_direct.sum(), "agent-hours")
print("scaled  :", s_scaled.tolist(), 0.5 * s_scaled.sum(), "agent-hours")
truth = models.call_centre_rate(mid) * models.CALL_CENTRE["monday_factor"]
for name, s in (("direct", s_direct), ("scaled", s_scaled)):
    ev = evaluate(s, truth, reps=30, seed0=500)
    print(f"{name}: tested against the true Monday profile - day {ev['day_sl']:.3f}, worst half hour {ev['slot_sl'].min():.3f}")

Monday factor 1.134 (s.e. about 0.017)
direct  : [9, 11, 13, 15, 15, 16, 15, 14, 13, 12, 10, 11, 12, 12, 12, 11, 10, 9, 7, 6, 6, 6, 6, 6] 128.5 agent-hours
scaled  : [9, 10, 13, 15, 15, 16, 15, 14, 12, 11, 12, 12, 12, 12, 11, 11, 10, 9, 7, 7, 6, 6, 6, 6] 128.5 agent-hours


direct: tested against the true Monday profile - day 0.943, worst half hour 0.888


scaled: tested against the true Monday profile - day 0.941, worst half hour 0.915


Four Mondays give half-hourly rates with standard errors near 15 %, so a schedule built on them inherits that
noise slot by slot; scaling the sixteen-day weekday profile by one well-estimated factor is far more precise.
Tested against the true Monday rate (known here because the data are synthetic) the scaled schedule is the
more reliable. Borrowing strength across days is the general lesson: estimate shape and level separately.

## Exercise 2

Patience is not in the data. Repeat the final assessment with mean patience 1, 3 and 10 minutes and with deterministic patience. How sensitive is the recommendation?

In [4]:
sched = lag_schedule(lam_week)
truth = models.call_centre_rate(mid)
rows = []
for name, pat in (("exponential, mean 1 min", lambda g: g.exponential(1 / 60)), ("exponential, mean 3 min", lambda g: g.exponential(3 / 60)),
                  ("exponential, mean 10 min", lambda g: g.exponential(10 / 60)), ("deterministic 3 min", lambda g: 3 / 60)):
    ev = evaluate(sched, truth, reps=20, seed0=700, patience=pat)
    rows.append((name, ev["day_sl"], ev["slot_sl"].min(), ev["day_ab"]))
print(pd.DataFrame(rows, columns=["patience", "service level", "worst half hour", "abandonment"]).to_string(index=False, float_format="%.3f"))

                patience  service level  worst half hour  abandonment
 exponential, mean 1 min          0.957            0.897        0.024
 exponential, mean 3 min          0.947            0.912        0.014
exponential, mean 10 min          0.941            0.866        0.006
     deterministic 3 min          0.933            0.875        0.001


Patience changes abandonment by a factor of several but the service level only modestly: impatient callers
leave, which shortens the queue for everyone else, so short patience *raises* the measured service level.
The recommendation (based on 20-second answering) is therefore robust, while any abandonment target would
need patience data - which the logs, recording only completed calls, do not contain.

## Exercise 3

**Implement it yourself:** add shift constraints - agents work 4-hour blocks starting on the hour - and find the cheapest set of shifts whose half-hourly coverage is at least the recommended schedule (a small integer program; brute force or a greedy heuristic), then assess it by simulation.

In [5]:
need = lag_schedule(lam_week)                        # required agents per half hour from 08:00
starts = np.arange(8.0, 16.01, 1.0)                    # 4-hour shifts starting on the hour, ending by 20:00
cover = np.array([[1 if s <= t0 < s + 4 else 0 for s in starts] for t0 in edges[:-1]])
shifts = np.zeros(len(starts), dtype=int)
while np.any(cover @ shifts < need):                   # greedy: add the shift covering the largest unmet need
    deficit = np.maximum(need - cover @ shifts, 0)
    shifts[np.argmax(cover.T @ deficit)] += 1
improved = True
while improved:                                        # drop redundant shifts
    improved = False
    for j in range(len(starts)):
        if shifts[j] > 0:
            shifts[j] -= 1
            if np.all(cover @ shifts >= need): improved = True
            else: shifts[j] += 1
print("shifts starting at", starts.tolist(), ":", shifts.tolist())
cov = cover @ shifts
print(f"agent-hours: shifts {4 * shifts.sum()}, required {0.5 * need.sum():.1f}; overstaffing in {np.sum(cov > need)} of {len(need)} half hours")
ev = evaluate(cov, models.call_centre_rate(mid), reps=40, seed0=900)
print(f"shift schedule on 40 fresh days: service level {ev['day_sl']:.3f}, worst half hour {ev['slot_sl'].min():.3f}")

shifts starting at [8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0] : [10, 3, 2, 1, 5, 3, 2, 1, 5]
agent-hours: shifts 128, required 116.0; overstaffing in 14 of 24 half hours


shift schedule on 40 fresh days: service level 0.965, worst half hour 0.901


Covering a half-hourly requirement with 4-hour shifts is a set-covering integer program; a greedy heuristic
(add the shift that covers the most unmet need, then remove redundant ones) gets close to optimal for small
instances like this. The shift structure costs extra agent-hours - the requirement curve has a sharp morning
peak that 4-hour blocks cannot follow - and the extra cover shows up as a higher service level. Split shifts or
part-timers are how real centres buy back the difference.